# K513 · Week 4, Session 2
## Overfitting and regularization — when a model is too big for its data

On Tuesday you built a model on 379 tracts and it missed by about \\$4,523 a tract. Today you take
almost all of that data away and watch what happens.

One question runs through the whole notebook:

> **When is a model too complicated for the data you actually have — and what do you do when you
> cannot get more data?**

Three sections:

| Section | The question |
|---|---|
| 1 | What happens to the same twelve columns when only 25 tracts have data on record? |
| 2 | Can a penalty on the coefficients buy back what the missing data cost? |
| 3 | Which of these models would you actually hand to a client? |

By the end you will have a habit — **the five steps to evaluate a model** — and a recommendation you
could defend in a meeting. The recommendation is the part we discuss.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

The specific trap in this session is that an AI will happily tell you which `alpha` is best. It will
pick the one that scored highest on the numbers you show it — which is precisely the mistake this
session is about. It cannot know that the number it is optimizing came from your test set, because
it cannot see where your numbers came from.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one. If
anything ever looks wrong: **Runtime → Restart session and run all**.

---
## 0 · Setup

Same libraries as Tuesday, plus two models: `Ridge` and `Lasso`. Both are linear regression with one
extra idea attached, and you will see how little the code changes.

Run all four cells in this section; only the last one shows anything.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option('display.precision', 3)

In [ ]:
BOSTON_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/BostonHousing.csv"

boston_df = pd.read_csv(BOSTON_URL)

X = boston_df.drop(columns=['MEDV'])
y = boston_df['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

print("tracts available to train on:", X_train.shape[0])
print("tracts held back to test on: ", X_test.shape[0])

### The helper

Everything today is one line, with two decisions in it: **how much data**, and **which model**.

`fit_and_score(n_tracts, model)` does five things you already know how to do — take a sample of that
many tracts, standardize the columns, fit the model, and score it on both the training rows and the
same 127 test tracts we have used all week.

Two things worth noticing before you run anything:

- The scaler is fitted on the **training rows only** and then used to transform both. That is the
  leakage rule from Week 3, and putting it inside a function is how you stop yourself forgetting it.
- `X_test` never influences anything. It is only ever scored against.

You do not need to be able to write this function. You do need to know what it is doing, because
every number in this notebook comes out of it.

In [ ]:
SEED = 15          # fixes which tracts are "on record", so everyone sees the same numbers


def fit_and_score(n_tracts, model, label=None):
    """Fit `model` on a sample of n_tracts, score it on train and on the held-out test set."""
    rng  = np.random.RandomState(SEED)
    rows = rng.choice(len(X_train), n_tracts, replace=False)
    X_small, y_small = X_train.iloc[rows], y_train.iloc[rows]

    scaler = StandardScaler().fit(X_small)
    model.fit(scaler.transform(X_small), y_small)

    train_r2 = model.score(scaler.transform(X_small), y_small)
    test_r2  = model.score(scaler.transform(X_test),  y_test)
    predicted = model.predict(scaler.transform(X_test))
    rmse = np.sqrt(mean_squared_error(y_test, predicted)) * 1000
    mae  = mean_absolute_error(y_test, predicted) * 1000
    kept = int((model.coef_ != 0).sum())

    name = label or f"{type(model).__name__}, {n_tracts} tracts"
    print(f"{name:<34} train R2 {train_r2:6.3f}   test R2 {test_r2:7.3f}"
          f"   RMSE ${rmse:>7,.0f}   MAE ${mae:>7,.0f}   columns used {kept}")
    return None


def baseline(n_tracts):
    """Predict the average of those n_tracts for every test row. The number to beat."""
    rng  = np.random.RandomState(SEED)
    rows = rng.choice(len(X_train), n_tracts, replace=False)
    guess = np.full(len(y_test), y_train.iloc[rows].mean())
    rmse = np.sqrt(mean_squared_error(y_test, guess)) * 1000
    mae  = mean_absolute_error(y_test, guess) * 1000
    print(f"{'Guess the average of ' + str(n_tracts):<34} "
          f"train R2    ---   test R2     ---   RMSE ${rmse:>7,.0f}   MAE ${mae:>7,.0f}")

---
## 1 · What happens with only 25 tracts?

You are valuing property in a town where only **25 tracts** have data on record. Same twelve
columns as Tuesday, a twentieth of the rows.

Before you run anything, commit to an answer:

- Will the **training** score go up, go down, or stay about the same?
- Will the **test** score go up, go down, or stay about the same?

### ✏️ Now You Try · 1

**(a)** Run the cell below. It reproduces Tuesday's model, so you know the helper is working — you
should get a training R² of **0.736**.

In [ ]:
fit_and_score(379, LinearRegression(), label="12 columns, 379 tracts")

**(b)** Now take the data away. Fill in the blank with the number of tracts on record.

In [ ]:
fit_and_score(____, LinearRegression(), label="12 columns, only 25 tracts")

**(c)** Add the baseline, so you know what "good" would even mean here.

In [ ]:
baseline(25)

**(d)** Write your answer in this cell, in your own words.

Which of the two scores on the 25-tract row is the bad news, and how would you explain what has
happened to someone who has not taken this course? Two or three sentences.

*(An answer that only says "the test score is low" has not explained anything. Say why the training
score being high is part of the problem.)*

> **Your answer:**
>
>

---
### Steps to Evaluate a Model

The habit this session exists to build. You will use it for the rest of the course.

1. **Score on train and on test.** Always both, always in that order.
2. **Both low?** Too simple. Give it more to work with.
3. **Train high, test far below?** Too much model for the data you have. Get more data, or add a
   penalty.
4. **Close together, and better than the baseline?** Stop. This is what this data has.
5. **Never** choose a setting because it scored best on the test set.

Section 1 was step 3. The rest of the notebook is what to do about it.

---
## 2 · Give the model a second objective

The first answer to step 3 is always **get more data**. Run this cell to see what that would be
worth — the same twelve columns, with more and more tracts on record.

In [ ]:
for n in [15, 25, 40, 60, 100, 200, 379]:
    fit_and_score(n, LinearRegression(), label=f"{n} tracts, no penalty")

In business the answer to *"can I have more data?"* is usually no. The data has not been collected, the
product is new, the quarter is the quarter.

So instead we change what the model is trying to do.

- **Before:** one objective — make the error on the rows in front of it as small as possible. It
  will do anything to achieve that, including memorizing twenty-five tracts.
- **After:** two objectives — make the error small, **and** keep the coefficients small. Now it has
  to trade them off.

`Ridge` is a linear regression with that second objective attached. `alpha` is how hard you lean on
it — the penalty dial. `alpha=0` is no penalty at all; turn it up and the coefficients are squeezed
toward zero.

### ✏️ Now You Try · 2

**(a)** Fill in the three alphas and run the cell. Report all three rows — do not delete the ones
you like less.

In [ ]:
fit_and_score(25, LinearRegression(), label="25 tracts, no penalty")
fit_and_score(25, Ridge(alpha=____),  label="25 tracts, alpha 1")
fit_and_score(25, Ridge(alpha=____),  label="25 tracts, alpha 20")
fit_and_score(25, Ridge(alpha=____),  label="25 tracts, alpha 500")

**(b)** What happens to the **training** score as alpha goes up? Why is that the expected direction
rather than a sign something has gone wrong? One or two sentences.

> **Your answer:**
>

**(c)** Which alpha gives the best **test** score, and by how many dollars of RMSE does it beat the
no-penalty model?

> **Your answer:**
>

**(d)** Run the evaluation steps on the **alpha = 500** row. Which of the five steps are you on, and what does
that step tell you to do?

> **Your answer:**
>

### Why you cannot just pick the winner

The cell below repeats the whole thing on eight *different* sets of 25 tracts from the same town, and
reports which alpha came first each time.

Run it, then read the row of winners.

In [ ]:
alphas = [0.1, 0.5, 1, 2, 5, 10, 20, 50, 100]
winners = []

for sample_seed in [15, 3, 7, 11, 22, 5, 19, 27]:
    rng  = np.random.RandomState(sample_seed)
    rows = rng.choice(len(X_train), 25, replace=False)
    X_small, y_small = X_train.iloc[rows], y_train.iloc[rows]
    scaler = StandardScaler().fit(X_small)

    scores = [Ridge(alpha=a).fit(scaler.transform(X_small), y_small)
                            .score(scaler.transform(X_test), y_test) for a in alphas]
    winners.append(alphas[int(np.argmax(scores))])

print("winning alpha on each of eight samples:", winners)

A tenfold spread, decided entirely by which twenty-five tracts happened to be on record.

So *"alpha = 2 is best"* was never a fact about tracts. It was a fact about that test set.

**Report the whole curve. Do not report the winner.** The moment you pick a setting because it
scored best on the test set, that score has stopped being an estimate of anything — which is
evaluation step 5.

---
## 3 · Which model would you actually ship?

`Lasso` is the same idea as `Ridge` with one difference that a manager would care about: it can set
a coefficient to **exactly zero**, so it also decides which columns you keep.

Fewer columns means fewer data feeds to maintain, fewer things to explain, and a model somebody can
hold in their head.

### ✏️ Now You Try · 3

**(a)** Fill in the two alphas and run it. Watch the last column — how many columns each model
actually used.

In [ ]:
fit_and_score(25, Lasso(alpha=____), label="25 tracts, LASSO alpha 1")
fit_and_score(25, Lasso(alpha=____), label="25 tracts, LASSO alpha 3")

**(b)** Which columns survived? Run this to find out.

In [ ]:
rng  = np.random.RandomState(SEED)
rows = rng.choice(len(X_train), 25, replace=False)
X_small, y_small = X_train.iloc[rows], y_train.iloc[rows]
scaler = StandardScaler().fit(X_small)

lasso = Lasso(alpha=1).fit(scaler.transform(X_small), y_small)

survivors = pd.Series(lasso.coef_, index=X.columns)
print(survivors[survivors != 0].sort_values(key=abs, ascending=False).round(2))

**(c)** Put all four models on one board so you can compare them.

In [ ]:
baseline(25)
fit_and_score(25,  LinearRegression(), label="25 tracts, no penalty")
fit_and_score(25,  Ridge(alpha=20),    label="25 tracts, Ridge")
fit_and_score(25,  Lasso(alpha=1),     label="25 tracts, LASSO")
fit_and_score(379, LinearRegression(), label="379 tracts, no penalty (Tuesday)")

**(d)** **This is the one we discuss.**

A lender needs a valuation model by Friday. They have **25 tracts on record** in this town and cannot
get more before then.

Which of the four models above do you hand them, and what do you tell them it **cannot** do? Four or
five sentences, written for someone who does not code.

There is no single right answer. What matters is whether your reasons match the numbers in your
own table — and whether you say something honest about the limits of a model built on twenty-five
tracts.

> **Your answer:**
>
>
>

---
### Before you close this notebook

Photograph the evaluation steps if you have not already. You will use it on tree depth in Week 5 and on every
model comparison in Week 6, and the steps do not change.

The exit ticket is on the bottom half of your card:

> In one sentence: a colleague shows you a model with a training R² of 0.98 and does not mention a
> test score. What do you ask for, and why?